# Motor - finger tapping features + Balanced RF (3 classes)

**Spec:** PLAN.md B4. **Runs on:** Colab. Upload `motor_code.tar` (`common/`, `motor/`, tiny) from `D:\Alzheimers\colab_upload\` to `/content` each session. Not run yet.

No labelled tapping data is available to us (FINDINGS 2). Cell 2 checks that the feature code works on a made-up session; **it is not data and shows no performance**. Cells 3-4 only run if you ever obtain real files, laid out as `DATA/<subject_id>.csv` (PsychoPy output of the reference task) plus `DATA/labels.csv` with columns `subject,group` where group is CN, aMCI or AD.

Upload the repo's `common/` and `motor/` folders. Set `REPO`.

In [ ]:
!pip -q install imbalanced-learn
import os, sys, subprocess
os.makedirs('/content/Alzheimers', exist_ok=True)
subprocess.run(['tar', '-xf', '/content/motor_code.tar', '-C', '/content/Alzheimers'], check=True)
import numpy as np, pandas as pd
REPO = '/content/Alzheimers'          # folder containing common/ and motor/
DATA = '/content/data/finger_tapping' # only used if you ever obtain real data
sys.path.insert(0, REPO)
from motor.features import session_features, FEATURE_NAMES
from motor.loader import trials_from_csv
from motor.cv import evaluate, CLASSES
from common.runlog import RunLog
log = RunLog('/content/out/motor.log')   # appends to a text file on Colab's temporary disk


## 2. Code check on a made-up session (not data)

In [ ]:
with log.tee('synthetic check'):
    r = [(0.4 + 0.25 * i, 'rshift') for i in range(38)]
    d = [(t, k) for i in range(18) for t, k in ((0.5 + 0.5 * i, 'lshift'), (0.52 + 0.5 * i, 'rshift'))]
    a = [(0.5 + 0.3 * i, 'lshift' if i % 2 == 0 else 'rshift') for i in range(30)]
    s = session_features({'right': [r, r, r], 'dual': [d, d, d], 'alt': [a, a, a]})
    print(len(s), 'features')
pd.Series(s).round(3)


> **SEND BACK TO CLAUDE:**
> Not needed now - no motor data exists. Skip this notebook for the moment.

## 3. Load real data (only if present)

In [ ]:
with log.tee('load real data'):
    lab = pd.read_csv(f'{DATA}/labels.csv')
    rows = [session_features(trials_from_csv(f'{DATA}/{s}.csv')) for s in lab.subject]
    X = pd.DataFrame(rows)[FEATURE_NAMES].values
    y = lab.group.map({c: i for i, c in enumerate(CLASSES)}).values
    print(X.shape, np.bincount(y), 'missing cells:', int(np.isnan(X).sum()))


## 4. Cross-validated result
Report only what this prints. Not comparable to the paper's 70% / 0.76 (different protocol).

In [ ]:
with log.tee('cross-validated result'):
    for r in evaluate(X, y): print(r)


## 5. Download the log
Downloads the log to your PC; move it to `D:\Alzheimersesults\`.


In [ ]:
# One zip = one download (browsers block several downloads at once)
import shutil
from google.colab import files
shutil.make_archive('/content/motor_results', 'zip', '/content/out')
print('Click Allow if the browser asks about downloads. If nothing downloads: Files panel > refresh > right-click motor_results.zip > Download.')
files.download('/content/motor_results.zip')
